In [1]:
import pandas as pd
from urllib.parse import urlparse

CSV_PATH = "../dataset_crawl.csv"

df = pd.read_csv(CSV_PATH)
print(f"Total baris awal : {len(df):,}")
print(f"Total domain     : {df['Domain'].nunique():,}")
df.head(3)

Total baris awal : 110,703
Total domain     : 685


,Webpage_id,Domain,Url,Tag,Crawled_at
0,1,asus.com,https://asus.com/id,non-gambling,2026-04-27 15:15:17
1,2,lazada.co.id,https://pages.lazada.co.id/wow/gcp/route/lazad...,non-gambling,2026-04-27 15:15:17
2,3,lazada.co.id,https://pages.lazada.co.id/wow/i/id/idcampaign...,non-gambling,2026-04-27 15:15:17


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Fungsi Filter (FINAL)
# ═══════════════════════════════════════════════════════════════
from urllib.parse import urlparse

MAX_URL_LENGTH = 300

def has_undefined_segment(url: str) -> bool:
    try:
        return any(
            s.lower() == "undefined" or s.lower() == "undefinedp"
            for s in urlparse(url).path.split("/") if s
        )
    except Exception:
        return False

def has_path_repetition(url: str) -> bool:
    try:
        segs = [s.lower() for s in urlparse(url).path.split("/") if s]
        seen_bigrams: set = set()
        for i in range(1, len(segs)):
            if segs[i] == segs[i - 1]:
                return True
            bg = (segs[i - 1], segs[i])
            if bg in seen_bigrams:
                return True
            seen_bigrams.add(bg)
        return False
    except Exception:
        return False

def is_url_dirty(url: str) -> tuple:
    url = str(url)
    if len(url) > MAX_URL_LENGTH:
        return True, "too-long"
    if has_undefined_segment(url):
        return True, "undefined-segment"
    if has_path_repetition(url):
        return True, "path-repetition"
    return False, ""

print("Fungsi loaded.")
print(f"  id has_undefined_segment : {id(has_undefined_segment)}")
print(f"  id has_path_repetition   : {id(has_path_repetition)}")
print(f"  id is_url_dirty          : {id(is_url_dirty)}")

Fungsi loaded.
  id has_undefined_segment : 128968449573888
  id has_path_repetition   : 128968449573728
  id is_url_dirty          : 128968449574208


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Unit Test (WAJIB HIJAU SEMUA SEBELUM LANJUT)
# ═══════════════════════════════════════════════════════════════
cases = [
    # --- harus REJECT ---
    ("https://www.samsclub.com/account/irr/irr/irr/launch",              "REJECT"),
    ("https://www.samsclub.com/shop/irr/irr/irr/irr/irr/irr/irr/launch","REJECT"),
    ("https://t30.tt88jaya.com/forgot/https/x.com/https/x.com/toto88",  "REJECT"),
    ("https://www.gotix.id/undefinedp/undefinedp/privacy-policy",        "REJECT"),
    ("https://www.ralali.com/c/undefined",                                "REJECT"),
    ("https://informa.co.id/home-promotion/undefined",                    "REJECT"),
    ("https://bbc.com/undefined",                                         "REJECT"),

    # --- harus ALLOW ---
    ("https://blog.rust-lang.org/2026/04/handling-undefined-symbols",    "ALLOW"),
    ("https://docs.python.org/3/library/urllib.parse.html",              "ALLOW"),
    ("https://tokopedia.com/elektronik/laptop/asus-vivobook-pro",        "ALLOW"),
    ("https://shop.example.com/category/sub/brand/product/variant",      "ALLOW"),
]

all_pass = True
for url, expected in cases:
    dirty, reason = is_url_dirty(url)
    result = "REJECT" if dirty else "ALLOW"
    ok = result == expected
    if not ok:
        all_pass = False
    icon = "✓" if ok else "✗ SALAH"
    print(f"{icon}  {result:<6}  [{reason or '-':<20}]  {url[:65]}")

print()
print("══ SEMUA PASS ══" if all_pass else "══ ADA YANG GAGAL — JANGAN LANJUT ══")

✓  REJECT  [path-repetition     ]  https://www.samsclub.com/account/irr/irr/irr/launch
✓  REJECT  [path-repetition     ]  https://www.samsclub.com/shop/irr/irr/irr/irr/irr/irr/irr/launch
✓  REJECT  [path-repetition     ]  https://t30.tt88jaya.com/forgot/https/x.com/https/x.com/toto88
✓  REJECT  [undefined-segment   ]  https://www.gotix.id/undefinedp/undefinedp/privacy-policy
✓  REJECT  [undefined-segment   ]  https://www.ralali.com/c/undefined
✓  REJECT  [undefined-segment   ]  https://informa.co.id/home-promotion/undefined
✓  REJECT  [undefined-segment   ]  https://bbc.com/undefined
✓  ALLOW   [-                   ]  https://blog.rust-lang.org/2026/04/handling-undefined-symbols
✓  ALLOW   [-                   ]  https://docs.python.org/3/library/urllib.parse.html
✓  ALLOW   [-                   ]  https://tokopedia.com/elektronik/laptop/asus-vivobook-pro
✓  ALLOW   [-                   ]  https://shop.example.com/category/sub/brand/product/variant

══ SEMUA PASS ══


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Debug manual (jalankan jika cell 3 ada yang gagal)
# ═══════════════════════════════════════════════════════════════
target = "https://www.samsclub.com/account/irr/irr/irr/launch"

segs = [s.lower() for s in urlparse(target).path.split("/") if s]
print("segments :", segs)

seen = set()
for i in range(1, len(segs)):
    print(f"  i={i}  prev={segs[i-1]!r}  curr={segs[i]!r}", end="  ")
    if segs[i] == segs[i - 1]:
        print("→ CONSECUTIVE → REJECT")
        break
    bg = (segs[i - 1], segs[i])
    if bg in seen:
        print(f"→ BIGRAM {bg} REPEAT → REJECT")
        break
    seen.add(bg)
    print(f"bigram {bg} added")

print()
print("has_path_repetition :", has_path_repetition(target))
print("is_url_dirty        :", is_url_dirty(target))

segments : ['account', 'irr', 'irr', 'irr', 'launch']
  i=1  prev='account'  curr='irr'  bigram ('account', 'irr') added
  i=2  prev='irr'  curr='irr'  → CONSECUTIVE → REJECT

has_path_repetition : True
is_url_dirty        : (True, 'path-repetition')


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Deteksi & Preview (jalankan setelah cell 3 semua ✓)
# ═══════════════════════════════════════════════════════════════
results      = df["Url"].apply(is_url_dirty)
df["_dirty"]  = results.apply(lambda x: x[0])
df["_reason"] = results.apply(lambda x: x[1])

dirty_df = df[df["_dirty"]].copy()

print(f"Total baris kotor : {len(dirty_df):,}")
print(f"Akan tersisa      : {(~df['_dirty']).sum():,} baris")
print()
print("Breakdown per alasan:")
print(dirty_df["_reason"].value_counts().to_string())
print()

print("Sampel per alasan:")
for reason in dirty_df["_reason"].unique():
    sub = dirty_df[dirty_df["_reason"] == reason][["Domain", "Url"]].head(3)
    print(f"\n--- {reason} ---")
    for _, row in sub.iterrows():
        preview = row["Url"][:100] + ("..." if len(row["Url"]) > 100 else "")
        print(f"  {row['Domain']}  |  {preview}")

Total baris kotor : 1,887
Akan tersisa      : 108,816 baris

Breakdown per alasan:
_reason
path-repetition      1571
too-long              240
undefined-segment      76

Sampel per alasan:

--- path-repetition ---
  uniqlo.com  |  https://uniqlo.com/id/id
  uniqlo.com  |  https://www.uniqlo.com/id/id/member
  asus.com  |  https://iot.asus.com/ai-software-ai-platforms/ai-software-ai-platforms/filter

--- undefined-segment ---
  bbc.com  |  https://bbc.com/undefined
  smarttask.io  |  https://smarttask.io/blog/category/undefined
  nuxt.com  |  https://nuxt.com/enterprise/agencies/undefined

--- too-long ---
  unicef.org  |  https://unicef.org/ar/%d8%a8%d9%84%d8%ba-%d8%b9%d8%af%d8%af-%d8%a7%d9%84%d8%ae%d8%b3%d8%a7%d8%a6%d8%...
  unicef.org  |  https://unicef.org/ar/%d8%aa%d8%b9%d9%85%d9%91%d9%8f%d9%82-%d8%a7%d9%84%d8%a3%d8%b2%d9%85%d8%a9-%d8%...
  unicef.org  |  https://unicef.org/ar/%d8%a8%d9%8a%d8%a7%d9%86-%d8%b5%d8%a7%d8%af%d8%b1-%d8%b9%d9%86-%d8%a7%d9%84%d9...


In [6]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Bersihkan & Simpan
# ═══════════════════════════════════════════════════════════════
OUTPUT_CSV  = "dataset_crawl_clean.csv"
OUTPUT_JSON = "dataset_crawl_clean.json"

df_clean = (
    df[~df["_dirty"]]
    .drop(columns=["_dirty", "_reason"])
    .sort_values("Webpage_id")
    .reset_index(drop=True)
)
df_clean["Webpage_id"] = df_clean.index + 1

df_clean.to_csv(OUTPUT_CSV, index=False)
df_clean.to_json(OUTPUT_JSON, orient="records", indent=2)

df.drop(columns=["_dirty", "_reason"], inplace=True)

print(f"Dihapus  : {len(df) - len(df_clean):,} baris")
print(f"Tersisa  : {len(df_clean):,} baris ({df_clean['Domain'].nunique():,} domain)")
print(f"CSV      -> {OUTPUT_CSV}")
print(f"JSON     -> {OUTPUT_JSON}")

Dihapus  : 1,887 baris
Tersisa  : 108,816 baris (685 domain)
CSV      -> dataset_crawl_clean.csv
JSON     -> dataset_crawl_clean.json


In [7]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Verifikasi akhir
# ═══════════════════════════════════════════════════════════════
check_urls = [
    "https://www.samsclub.com/account/irr/irr/irr/launch",
    "https://www.gotix.id/undefinedp/undefinedp/privacy-policy",
    "https://informa.co.id/home-promotion/undefined",
]

for u in check_urls:
    found = df_clean["Url"].str.contains(u, regex=False, na=False).sum()
    status = "✓ BERSIH" if found == 0 else f"✗ MASIH ADA ({found} baris)"
    print(f"{status}  {u[:70]}")

print()
print("Distribusi tag:")
print(df_clean["Tag"].value_counts().to_string())

✓ BERSIH  https://www.samsclub.com/account/irr/irr/irr/launch
✓ BERSIH  https://www.gotix.id/undefinedp/undefinedp/privacy-policy
✓ BERSIH  https://informa.co.id/home-promotion/undefined

Distribusi tag:
Tag
non-gambling    101313
gambling          7503
